# 🧬 Práctica: análisis de datos de nanoporo en la nube
**Curso de Análisis Bioinformático de Datos Genómicos aplicado al Diagnóstico Genético · Bloque II (Long-Read Sequencing)**
Belén de la Morena Barrio · CIBERER / Longseq Applications

---

### ¿Qué vas a hacer?
Partirás de las **lecturas crudas (FASTQ)** de un paciente secuenciado con nanoporo y recorrerás el pipeline completo que hemos visto en clase:

| Paso | Herramienta | Fichero que obtienes |
|---|---|---|
| 1. Control de calidad | Python | gráficas + métricas (N50, Q media) |
| 2. Alineamiento al genoma de referencia | **minimap2** + samtools | `.bam` + `.bai` |
| 3. Cobertura en la región de interés | samtools | tabla |
| 4. Llamada de variantes estructurales | **Sniffles2** | `.vcf` |
| 5. Visualización | **IGV** | ¡tus propios ojos! 👀 |

### Cómo se usa este cuaderno
* **No tienes que instalar nada.** Todo corre en un ordenador de Google, gratis.
* Ejecuta las celdas **en orden, de arriba abajo**: pulsa ▶ a la izquierda de cada celda o `Mayús + Intro`.
* Si algo falla: `Entorno de ejecución → Reiniciar y ejecutar todo`.
* **En Google Colab:** haz primero **`Archivo → Guardar una copia en Drive`** para trabajar sobre tu propia copia.
* **En GitHub Codespaces:** arriba a la derecha pulsa **«Select Kernel» → Python Environments → Python 3** la primera vez.

> 🎯 **El reto:** este paciente tiene un diagnóstico clínico claro, pero Sanger, NGS y MLPA salieron **negativos**. Tu misión es encontrar la causa genética.

## 0. Parámetros del ejercicio
No hace falta tocarlos (la profesora ya los ha rellenado), pero fíjate en qué significa cada uno.

In [ ]:
#@title Parámetros { display-mode: "form" }
# ID del FASTQ compartido en Google Drive (lo que va después de /d/ en el enlace)
FASTQ_DRIVE_ID = "P1_subset_bioinfo"  #@param {type:"string"}
MUESTRA = "paciente1"  #@param {type:"string"}
# Cromosoma donde está la región enriquecida (adaptive sampling)
CROMOSOMA = "chr1"  #@param {type:"string"}
# Región / gen de interés para inspeccionar (coordenadas hg38)
REGION = "chr1:173900000-173950000"  #@param {type:"string"}
GEN = "SERPINC1"  #@param {type:"string"}
# ¿Las lecturas llevan etiquetas de metilación (MM/ML) de Dorado? (avanzado)
METILACION = False  #@param {type:"boolean"}

import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    TRABAJO = "/content/practica"
else:  # GitHub Codespaces u otro Jupyter
    TRABAJO = os.path.join(os.environ.get("CODESPACE_VSCODE_FOLDER", os.path.expanduser("~")), "resultados")
os.makedirs(TRABAJO, exist_ok=True)
os.chdir(TRABAJO)
print("Entorno:", "Google Colab" if EN_COLAB else "GitHub Codespaces / Jupyter")
print("Trabajando en", TRABAJO)

## 1. Instalación de las herramientas (~1 min)
Instalamos **minimap2** (alineador), **samtools** (manejo de BAM), **Sniffles2** (variantes estructurales) e **igv-notebook** (IGV dentro del cuaderno).

In [ ]:
%%capture instalacion
import shutil
if EN_COLAB:
    !apt-get -qq update
    !apt-get -qq install -y minimap2 samtools tabix
    !pip -q install sniffles==2.4 igv-notebook gdown
elif not shutil.which("minimap2"):  # en Codespaces normalmente ya vienen instaladas
    !sudo apt-get -qq update
    !sudo apt-get -qq install -y minimap2 samtools tabix zip
    !pip -q install sniffles==2.4 gdown pandas matplotlib

In [ ]:
!minimap2 --version && samtools --version | head -1 && sniffles --version 2>/dev/null | tail -1

## 2. Descarga de los datos (~2 min)
* **FASTQ** del paciente (compartido por la profesora en Google Drive).
* **Genoma de referencia**: solo el cromosoma de interés del hg38 (para ir más rápido), desde UCSC.

In [ ]:
import gdown, subprocess
# Vale tanto un FASTQ sin comprimir (.fastq) como comprimido (.fastq.gz)
if not any(os.path.exists(f"{MUESTRA}{e}") for e in (".fastq", ".fastq.gz")):
    gdown.download(id=FASTQ_DRIVE_ID, output="descarga.tmp", quiet=False)
    with open("descarga.tmp", "rb") as fh:
        comprimido = fh.read(2) == b"\x1f\x8b"
    os.rename("descarga.tmp", f"{MUESTRA}.fastq.gz" if comprimido else f"{MUESTRA}.fastq")
FASTQ = f"{MUESTRA}.fastq.gz" if os.path.exists(f"{MUESTRA}.fastq.gz") else f"{MUESTRA}.fastq"
print("FASTQ:", FASTQ)

REF = f"{CROMOSOMA}.fa"
if not os.path.exists(REF):
    !wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/{CROMOSOMA}.fa.gz
    !gunzip -f {CROMOSOMA}.fa.gz
    !samtools faidx {REF}
!ls -lh

❓ **Pregunta 1.** Abre el FASTQ y mira las 4 primeras líneas. ¿Qué hay en cada una? ¿Qué codifica la cuarta?

In [ ]:
!zcat -f {FASTQ} 2>/dev/null | head -4 | cut -c1-150

## 3. Control de calidad de las lecturas

In [ ]:
import gzip, math, statistics
import matplotlib.pyplot as plt

def leer_fastq(path):
    with open(path, "rb") as fh:
        es_gz = fh.read(2) == b"\x1f\x8b"
    op = gzip.open if es_gz else open
    with op(path, "rt") as f:
        while True:
            h = f.readline()
            if not h: break
            s = f.readline().strip(); f.readline(); q = f.readline().strip()
            yield s, q

longitudes, calidades = [], []
for s, q in leer_fastq(FASTQ):
    longitudes.append(len(s))
    # Q media correcta: se promedian probabilidades de error, no valores Phred
    p = sum(10 ** (-(ord(c) - 33) / 10) for c in q) / len(q)
    calidades.append(-10 * math.log10(p))

orden = sorted(longitudes, reverse=True); total = sum(orden); acum = 0
for L in orden:
    acum += L
    if acum >= total / 2: n50 = L; break

print(f"Lecturas:            {len(longitudes):,}")
print(f"Bases totales:       {total/1e6:,.1f} Mb")
print(f"Longitud media:      {statistics.mean(longitudes):,.0f} pb")
print(f"N50:                 {n50:,} pb")
print(f"Lectura más larga:   {max(longitudes):,} pb")
print(f"Q media (mediana):   {statistics.median(calidades):.1f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(longitudes, bins=60, color="#2b6cb0"); ax[0].set_xlabel("Longitud de lectura (pb)"); ax[0].set_ylabel("Nº lecturas")
ax[0].axvline(n50, color="crimson", ls="--", label=f"N50 = {n50:,}"); ax[0].legend()
ax[1].hist(calidades, bins=40, color="#38a169"); ax[1].set_xlabel("Calidad media por lectura (Phred)")
plt.tight_layout(); plt.show()

❓ **Pregunta 2.** ¿Qué es el N50? Compara la longitud de estas lecturas con las de Illumina (2×150 pb). ¿Qué tipo de variantes podrás ver ahora que antes no?

## 4. Alineamiento contra el genoma de referencia (minimap2)
`-ax map-ont` = preset para lecturas de nanoporo. El resultado se ordena e indexa con samtools → **BAM + BAI**.

In [ ]:
BAM = f"{MUESTRA}.bam"
copiar_tags = "-y" if METILACION else ""   # -y conserva las etiquetas MM/ML de metilación
!minimap2 -t 2 -ax map-ont {copiar_tags} {REF} {FASTQ} 2> minimap2.log | samtools sort -@2 -o {BAM} -
!samtools index {BAM}
!samtools flagstat {BAM} | head -7

❓ **Pregunta 3.** ¿Qué porcentaje de lecturas ha alineado? ¿Qué significan las lecturas *supplementary*? (pista: piensa en una lectura que cruza una translocación o una inversión)

## 5. Cobertura en la región de interés

In [ ]:
!samtools coverage -r {REGION} {BAM}
!samtools coverage -r {REGION} -m {BAM} | tail -n +2 | head -20

❓ **Pregunta 4.** ¿Cuál es la profundidad media en la región? Este experimento usó **adaptive sampling**: ¿cómo lo notas al comparar con el resto del cromosoma?

In [ ]:
!samtools coverage {BAM}

## 6. Llamada de variantes estructurales (Sniffles2)

In [ ]:
VCF = f"{MUESTRA}.sv.vcf"
!sniffles --input {BAM} --vcf {VCF} --reference {REF} --threads 2 2>&1 | tail -1
!bgzip -kf {VCF} && tabix -f -p vcf {VCF}.gz

import pandas as pd
filas = []
for linea in open(VCF):
    if linea.startswith("#"): continue
    c = linea.rstrip("\n").split("\t")
    info = dict(kv.split("=", 1) if "=" in kv else (kv, True) for kv in c[7].split(";"))
    filas.append(dict(CROM=c[0], POS=int(c[1]), ID=c[2], TIPO=info.get("SVTYPE"),
                      TAMAÑO=info.get("SVLEN"), LECTURAS=info.get("SUPPORT"),
                      GENOTIPO=c[9].split(":")[0], FILTRO=c[6], ALT=c[4]))
svs = pd.DataFrame(filas)
print(f"Variantes estructurales detectadas en {CROMOSOMA}: {len(svs)}")
crom, rango = REGION.split(":"); ini, fin = map(int, rango.replace(",", "").split("-"))
en_region = svs[(svs.CROM == crom) & (svs.POS.between(ini, fin))]
print(f"En la región de {GEN} ({REGION}): {len(en_region)}")
en_region.drop(columns="ALT")

❓ **Pregunta 5.** ¿Qué tipo de variante encuentras en el gen? ¿De qué tamaño? ¿Está en homocigosis o heterocigosis? ¿Cuántas lecturas la soportan?

### 6b. ¿Qué es la secuencia insertada?
Guardamos la secuencia de las inserciones de la región en un FASTA. Cópiala y pégala en **[Dfam](https://dfam.org/search/sequence)** (base de datos de elementos repetitivos) o en **[BLAST](https://blast.ncbi.nlm.nih.gov/)**.

In [ ]:
ins = en_region[(en_region.TIPO == "INS") & (en_region.ALT.str.len() > 50)]
with open("inserciones.fa", "w") as f:
    for _, r in ins.iterrows():
        f.write(f">{r.ID}_{r.CROM}:{r.POS}_{len(r.ALT)}pb\n{r.ALT}\n")
!cat inserciones.fa | cut -c1-2000

❓ **Pregunta 6.** ¿Qué tipo de elemento es? ¿Cómo ha llegado ahí? (recuerda la diapositiva de retrotransposones: L1, Alu, **SVA**…). ¿Por qué ninguna de las técnicas previas lo detectó?

## 7. Visualización en IGV (dentro del cuaderno)
Se abre un IGV aquí mismo con tu BAM y tu VCF. Muévete, haz zoom y busca la variante.

💡 Consejos: activa *“Show soft clips”*, colorea las lecturas por *“insertion size”* y fíjate en las **marcas moradas** (inserciones) en las lecturas.

In [ ]:
if not EN_COLAB:
    print("En Codespaces IGV no se abre dentro del cuaderno: pasa directamente al paso 8 y usa igv.org/app.")
else:
  import igv_notebook
  igv_notebook.init()
  b = igv_notebook.Browser({"genome": "hg38", "locus": GEN})
  b.load_track({"name": f"{MUESTRA} SV", "path": f"{TRABAJO}/{VCF}.gz",
                "indexPath": f"{TRABAJO}/{VCF}.gz.tbi", "format": "vcf", "type": "variant"})
  b.load_track({"name": f"{MUESTRA} lecturas", "path": f"{TRABAJO}/{BAM}",
                "indexPath": f"{TRABAJO}/{BAM}.bai", "format": "bam", "type": "alignment",
                "showSoftClips": True, **({"colorBy": "basemod2"} if METILACION else {})})

## 8. Descarga tus resultados y ábrelos en IGV de escritorio
1. Ejecuta la celda: se descarga `resultados_<muestra>.zip`.
2. Abre **IGV Desktop** (o sin instalar nada: **[igv.org/app](https://igv.org/app)**), genoma **hg38**.
3. `File → Load from File…` y selecciona el `.bam` (con su `.bai` en la misma carpeta) y el `.vcf.gz`.
4. Busca el gen y haz una **captura de pantalla** de la variante: ¡es tu informe!

In [ ]:
!zip -q resultados_{MUESTRA}.zip {BAM} {BAM}.bai {VCF}.gz {VCF}.gz.tbi inserciones.fa
if EN_COLAB:
    from google.colab import files
    files.download(f"resultados_{MUESTRA}.zip")
else:
    print(f"Tu fichero está en: {TRABAJO}/resultados_{MUESTRA}.zip")
    print("Descárgalo: panel de la izquierda (Explorador) → carpeta 'resultados' → botón derecho sobre el .zip → Download")

## 🏁 Cierre
Comparte en el chat de clase:
1. Gen y posición de la variante · 2. Tipo y tamaño · 3. Qué elemento es · 4. Tu captura de IGV.

**Para pensar:** ¿Cómo validarías este hallazgo en el laboratorio? ¿Cómo lo describirías en nomenclatura HGVS?

---
*Extensión (si te sobra tiempo):* cambia `REGION` a otra zona del cromosoma y busca otras variantes estructurales. ¿Cuántas son polimorfismos comunes? (consulta gnomAD-SV).